# 💡 SQL vs. SQLAlchemy Core 메서드 매핑

| SQL 구문                 | SQLAlchemy Core 메서드              | 설명                                        |
| :----------------------- | :---------------------------------- | :------------------------------------------ |
| `CREATE DATABASE`        | `engine.execute("CREATE DATABASE")` | 데이터베이스 생성                            |
| `CREATE TABLE`           | `Table(...)`, `metadata.create_all(engine)` | 테이블 구조 정의 및 실제 테이블 생성         |
| `INSERT INTO ... VALUES` | `table.insert()`, `conn.execute(...)` | 테이블에 데이터 추가                         |
| `SELECT ... FROM ...`    | `select(...)`                       | 테이블에서 데이터 조회                       |
| `WHERE 조건`             | `.where(...)`                       | 특정 조건을 만족하는 데이터 필터링           |
| `ORDER BY 컬럼`          | `.order_by(...)`                    | 특정 컬럼 기준으로 데이터 정렬               |
| `LIMIT 숫자`             | `.limit(...)`                       | 조회 결과의 개수 제한                        |
| `COUNT(컬럼)`            | `func.count(table.c.column)`        | 특정 컬럼의 개수 세기 (집계 함수)            |
| `AVG(컬럼)`              | `func.avg(table.c.column)`          | 특정 컬럼의 평균 계산 (집계 함수)            |
| `GROUP BY 컬럼`          | `.group_by(...)`                    | 특정 컬럼 기준으로 데이터를 그룹화           |
| `HAVING 조건`            | `.having(...)`                      | 그룹화된 결과에 조건 필터링                  |
| `AS 별칭`                | `.label('별칭')`                    | 컬럼이나 집계 결과에 별명 지정               |
| `JOIN`                   | `.join(...)`                        | 여러 테이블을 연결하여 데이터 조회           |
| `UPDATE ... SET ... WHERE` | `update(...)`, `.where(...)`, `.values(...)` | 테이블의 특정 데이터 수정                    |
| `DELETE FROM ... WHERE`  | `delete(...)`, `.where(...)`        | 테이블에서 특정 데이터 삭제                  |
| `COMMIT`                 | `conn.commit()`                     | 데이터베이스 변경사항 최종 반영 (필수!!!)      |

# 🏠  나만의 운동 기록 및 루틴 분석 DB 만들기

## 🏗️ 기본 세팅

In [ ]:
# 1. MySQL 서버 설치 및 실행
!apt-get update
!apt-get install mysql-server > /dev/null
!service mysql start

# 2. 보안 설정 변경 (비밀번호 없이 root 접속 허용)
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY '';"

# 3. 필수 라이브러리 설치
!pip install ipython-sql pymysql sqlalchemy

# 4. SQL 매직 명령어 및 DB 접속 설정
%load_ext sql
%sql mysql+pymysql://root:@localhost/
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

print("✅ 숙제 준비 완료!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,144 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,151 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [92.7 kB]
Get:14 http://

## 📋 데이터 명세

**1. 환자 테이블 (patients)**
| id (PK) | name | age |
| :--- | :--- | :--- |
| 1 | 김오즈 | 34 |
| 2 | 이오즈 | 52 |
| 3 | 박오즈 | 29 |

**2. 건강 기록 테이블 (health_logs)**
| id (PK) | patient_id (FK) | blood_sugar | note |
| :--- | :--- | :--- | :--- |
| 1 | 1 | 95 | 식전 정상 |
| 2 | 3 | 145 | 주의: 식후 높음 |
| 3 | 2 | 110 | 양호 |

> 1. 표의 혈당 수치와 메모를 각 환자 ID에 맞게 객체로 생성해서 저장하세요.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, DateTime, ForeignKey, select, func, update, delete, text
from datetime import datetime

# 데이터베이스 생성 및 엔진 설정
engine = create_engine('mysql+pymysql://root:@localhost/')

with engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS health_log_db"))
    conn.commit()

engine = create_engine('mysql+pymysql://root:@localhost/health_log_db')
metadata = MetaData()


#환자 테이블 (patients)
patients = Table('patients', metadata,
    Column('patient_id', Integer, primary_key=True),
    Column('name', String(50), nullable=False),
    Column('Age', Integer)
)
#건강 기록 테이블 (health_logs)

health_log = Table('health_log', metadata,
    Column('health_log_id', Integer, primary_key=True),
    Column('patient_id', Integer, ForeignKey('patients.patient_id')),
    Column('blood_sugar', String(50)),
    Column('note', String(100))
)

metadata.create_all(engine)


# 데이터 적재

with engine.connect() as conn:
    conn.execute(patients.delete())
    conn.execute(health_log.delete())

    conn.execute(patients.insert(),[
        {'patient_id':1,'name':'김오즈','age':34},
        {'patient_id':2,'name':'이오즈','age':52},
        {'patient_id':3,'name':'박오즈','age':29}
    ])
    conn.execute(health_log.insert(),[
        {'health_log_id':1,'patient_id':1,'blood_sugar':95, 'note':'식전 정상'},
        {'health_log_id':2,'patient_id':3,'blood_sugar':145, 'note':'주의: 식후 높음'},
        {'health_log_id':3,'patient_id':2,'blood_sugar':110, 'note':'양호'}
    ])
    conn.commit()

    print("나만의 건강 일기장 DB 구축 & 테이블 생성 완료💚")

나만의 건강 일기장 DB 구축 & 테이블 생성 완료💚


> 2. 혈당이 120 이상인 사람의 메모(note)만 출력해보세요.

In [ ]:
from sqlalchemy import select

stmt_note = select(
    health_log.c.patient_id.label('환자ID'),
    health_log.c.note.label('혈당수치')
).where(health_log.c.blood_sugar >= 120)

with engine.connect() as conn:
    results= conn.execute(stmt_note)
    display(pd.DataFrame(results.fetchall(), columns=results.keys()))


,환자ID,혈당수치
0,3,주의: 식후 높음


> 3. 김오즈(patient_id = 1)의 혈당을 100으로 수정해보세요.

In [ ]:

stmt_updated = update(health_log
).where(health_log.c.patient_id == 1
).values(blood_sugar=100)

with engine.connect() as conn:
  conn.execute(stmt_updated)
  conn.commit()

  result_updated = conn.execute(select(health_log)).all()
  print("김오즈 혈당 수정 후 전체 데이터:",result_updated)

김오즈 혈당 수정 후 전체 데이터: [(1, 1, '100', '식전 정상'), (2, 3, '145', '주의: 식후 높음'), (3, 2, '110', '양호')]


> 4. 종합 분석 보고서 만들기: 모든 환자의 이름, 평균 혈당, 상태를 출력하세요.
>   * **상태 분류**: 평균 혈당이 110 이상이면 '주의', 아니면 '정상'으로 표시 (SQLAlchemy의 `case` 활용)
>   * 환자 테이블과 건강 기록 테이블을 **JOIN**하여 작성하세요.

In [ ]:
from sqlalchemy import case # case 함수를 사용하기 위해 import

# 상태 분류
stmt = select(
    patients.c.name,
    func.avg(health_log.c.blood_sugar).label('평균혈당'),
    case(
        (func.avg(health_log.c.blood_sugar) >= 110, '주의'),
        else_='정상'
    ).label('상태')
).select_from(
    patients.join(health_log, patients.c.patient_id == health_log.c.patient_id, isouter=True)
).group_by(patients.c.patient_id, patients.c.name) # name 컬럼과 patient_id 컬럼을 GROUP BY 절에 추가

with engine.connect() as conn:
    results= conn.execute(stmt)
    display(pd.DataFrame(results.fetchall(), columns=results.keys()))

,name,평균혈당,상태
0,김오즈,100.0,정상
1,이오즈,110.0,주의
2,박오즈,145.0,주의
